# Imputation and Synthetic Missingness

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np
import datetime as dt
from src.imputation import impute_ts
from src.missingness import simulate_missingness, create_mask

In [3]:
df = pd.read_csv(PROJECT_ROOT / "data" / "ETTm1.csv", parse_dates=["date"], index_col="date")

In [4]:
train = df.iloc[:1000]

# Pipeline

### Split data

`train, test = ...`

**Must normalise train to 0 1 scale so that errors can be aggregated**

### Simulate missingness

Create an empty dictionary and then iterate over the 10 loops, mechanisms, and missingness levels.

In [ ]:
missing_dict = {}
mask_dict = {}

# Iterate over the random seem so that each masks is unique and matches missingness in df.

for i in range(10):

    for mech in ["mar", "mcar", "mnar"]:

        for n in [10, 30, 50]:
    
            key = mech + str(n)

            if key not in mask_dict:
                mask_dict[key] = []
            
            mask_dict[key].append(create_mask(df=train, mechanism=mech, n=n, random_seed=i))

            if key not in missing_dict:
                missing_dict[key] = []

            missing_dict[key].append(simulate_missingness(df=train, mechanism=mech, n=n, random_seed=i))

`missing_dict` will be of format `{mcar10 : [df1, df2, ... , df10], mcar30 : [df1, ...], ... , mnar50 : [df1, ... , df10]}`.

`mask_dict` will be of format `{mcar10 : [mask1, mask2, ..., mask10] , ... , mnar50 : [mask1, ... , mask10]}`.

### Imputation

Iterate over the models and missing_dict dataframes.

In [8]:
models = ["locf", "ks", "arima", "lstm"]
imputed_data = {}

for model in models:

    for mech_prop, df_list in missing_dict.items():
        
        if model not in imputed_data:
            imputed_data[model] = {}

        imputed_data[model][mech_prop] = []

        for df in df_list:
            imputed_data[model][mech_prop].append(impute_ts(df, model))

`imputed_data` will be of format `{locf : {mcar10 : [df1, ..., df10], mcar30 : [df1, ..., df10], ..., mnar50 : [df1, ..., df10]}, ks : {}, ..., lstm : {}}`.

### Reconstruction evaluation

Need to store errors in a list, from which rmse, mape, bias and variance can be calculated for each model-mechanism-proportion combination.

Only calculate errors from imputed values, since values that were not imputed will have 0 error and affected calcs.

Subtract train from imputed and subset to mask, store in list.

`errors` will  be of format `{locf : {mar10 : [errors_list1, errors_list2, ..., errors_list10] , mar30 : [], ...}, ...}`

In [ ]:
errors = {}

for model in models:

    for mech_prop, df_list in imputed_data[model].items():

        if model not in errors:
            errors[model] = {}

        errors[model][mech_prop] = []

        for i, df in enumerate(df_list):
            
            error_array = (df - train).to_numpy()
            mask_array = mask_dict[mech_prop][i].to_numpy()


            errors[model][mech_prop].append(error_array[mask_array])

### Evaluation

For each model-mech-prop combination, calculate rmse and mae for each df then average across all 10 error lists.

Using MAE instead of MAPE as values are normalised.

In [ ]:
def calc_rmse(array:np.array):
    """
    Calculate RMSE from list of errors y_i - y_i-pred.
    """
    return np.sqrt(np.mean(array ** 2))

def calc_mae(array:np.array):
    """
    Calculate MAE from list of errors y_i - y_i-pred.
    """
    return np.mean(abs(array))

In [ ]:
reconstruction = {}

for model in models:

    for mech_prop, error_lists in errors[model].items():

        if model not in reconstruction:
            reconstruction[model] = {}

        rmse_list = []
        mae_list = []

        for i, error_list in enumerate(error_lists):

            rmse_list.append(calc_rmse(error_list))
            mae_list.append(calc_mae(error_list))
        
        if mech_prop not in reconstruction[model]:
            reconstruction[model][mech_prop] = {}
        
        reconstruction[model][mech_prop]["rmse"] = np.mean(rmse_list)
        reconstruction[model][mech_prop]["mae"] = np.mean(mae_list)

`reconstruction` is of format `{locf : {mar10 : {rmse : value, mae : value}, mar30 : {rmse : value, mae : value}, ...}, ...}`